# Phase 23 — Which Part of Topology Drives the Localization Benefit?

Phase 22 confirmed (8/8 seed-paired wins, p~0.008) that persistent homology
improves pocket-CENTER localization, not just classification. That is a
"that" finding. This notebook asks "why" — which specific piece of the
topology block is doing the work.

The topology feature block decomposes into four pieces with distinct
mathematical meanings:

- **H0** (connected components) — how fragmented / clustered the local point
  cloud is.
- **H1** (loops / tunnels) — 1-dimensional cycles, e.g. access channels.
- **H2** (enclosed voids) — 2-dimensional cavities that trap volume. This is
  the mathematically direct description of what a binding pocket *is*.
- **burial** (`nbhd_n_atoms`, `nbhd_gyration`) — density/depth, already shown
  in Phase 2b to be a real but separate signal from topology, for
  classification. Never tested for localization.

**Hypothesis, stated before running anything (so this is a real test, not a
post-hoc story):** if the "cavity" intuition for why topology helps is
correct, H2 alone should recover most of `full`'s localization advantage over
`no_topo`. If instead H1 or burial alone does the same, that points to
pockets being found via access channels or simple depth, not cavity shape —
a different and equally interesting mechanistic answer.

**Scope:** 3 seeds per new arm (H0-only, H1-only, H2-only, burial-only) — a
first-pass screen, same discipline as Phase 18's original single-pass topology
test before Phase 22 validated it further with 8 seeds. If a clear winner
emerges here, it earns the same multi-seed treatment next.

## Cell 1 — Setup, rebuild training data (Phase 14 logic, reused)

In [ ]:
!pip install -q torch-geometric==2.6.1 biopython==1.84

In [ ]:
import pickle

import gzip, json, os, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase10_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} | " +
      " | ".join(f"{k} {len(v)}" for k,v in splits.items()))

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_LIG_ATOMS = 6
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

need = sorted({d.pdb_id for d in dataset})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

def ligands_and_ca(pid, path, chain_id, n_expect):
    from Bio.PDB import MMCIFParser
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[chain_id]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues) != n_expect: return None
    raw_ca = np.asarray([r["CA"].coord for r in residues], dtype=np.float32)

    ligs = []
    for c2 in model:
        for res in c2:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                             dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS: ligs.append(xyz)
    return raw_ca, ligs

t0=time.time(); n_ok=0; n_geom_fail=0; n_no_ligand=0
for d in dataset:
    if d.pdb_id not in paths: continue
    r = ligands_and_ca(d.pdb_id, paths[d.pdb_id], d.chain_id, int(d.num_nodes))
    if r is None: continue
    raw_ca, ligs = r
    if not ligs: n_no_ligand += 1; continue

    # geometry-drift check: pairwise CA distances must match d.pos (residue ORDER, not just count)
    k = min(50, len(raw_ca))
    a = np.linalg.norm(raw_ca[:k,None]-raw_ca[None,:k], axis=-1)
    b = np.linalg.norm(d.pos.numpy()[:k,None]-d.pos.numpy()[None,:k], axis=-1)
    if float(np.abs(a-b).max()) > 1e-2:
        n_geom_fail += 1; continue

    offset = raw_ca.mean(axis=0)                       # same centering d.pos already used
    y_np = d.y.numpy().astype(bool)
    contact_centroid = raw_ca[y_np].mean(axis=0) if y_np.any() else raw_ca.mean(axis=0)
    # "primary" ligand = the one whose center of mass is nearest the labelled contact residues
    coms = [lig.mean(axis=0) for lig in ligs]
    best = int(np.argmin([np.linalg.norm(c - contact_centroid) for c in coms]))
    d.true_center = torch.from_numpy((coms[best] - offset).astype(np.float32))
    n_ok += 1

print(f"\ntrue centers recovered: {n_ok:,}/{len(dataset):,} "
      f"({(time.time()-t0)/60:.1f} min) | geometry mismatches: {n_geom_fail} | no ligand: {n_no_ligand}")

dataset = [d for d in dataset if hasattr(d, "true_center")]
print(f"training chains with recovered true centers: {len(dataset):,}")

rng = np.random.RandomState(42)
perm = rng.permutation(len(dataset))
n_val = max(1, int(0.08 * len(perm)))
splits = {"val": [dataset[i] for i in perm[:n_val]], "train": [dataset[i] for i in perm[n_val:]]}
n_tr, n_va = len(splits["train"]), len(splits["val"])
print(f"deployment training: {n_tr} train / {n_va} val  (matches Phase 15/18/22's setup)")

## Cell 2 — Model (identical to Phase 14/15/18/22)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 3 — Training machinery (identical to Phase 14)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)                    # ~matches the 4A DCC threshold
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=use_loc_head).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 4 — Build H0 / H1 / H2 / burial column masks

Reconstructs the same 35-name -> 34-kept mapping used in every phase that
touches this topology block, then further splits by homology dimension.
Matching by NAME (not a hardcoded index) so this self-verifies via the assert
rather than silently drifting.

In [ ]:
blob6b = torch.load(locate("phase6b_graphs.pt"), map_location="cpu", weights_only=False)
TOPO_NAMES = blob6b["topo_names"]
BURIAL_NAMES = set(blob6b["burial_names"])
print(f"topo columns: {len(TOPO_NAMES)} -- {TOPO_NAMES[:3]} ... {TOPO_NAMES[-2:]}")

MASKS = {
    "H0_only":     np.array([n.startswith("H0_") for n in TOPO_NAMES]),
    "H1_only":     np.array([n.startswith("H1_") for n in TOPO_NAMES]),
    "H2_only":     np.array([n.startswith("H2_") for n in TOPO_NAMES]),
    "burial_only": np.array([n in BURIAL_NAMES for n in TOPO_NAMES]),
}
for name, mask in MASKS.items():
    kept = [n for n,k in zip(TOPO_NAMES, mask) if k]
    print(f"  {name:<14} {mask.sum()} columns: {kept}")
assert sum(m.sum() for m in MASKS.values()) == len(TOPO_NAMES), \
    "masks should exactly partition the topo block (no overlap, no gaps)"
print("masks partition the full topo block exactly -- no overlap, no gaps")

## Cell 5 — Train 3 seeds per arm (H0/H1/H2/burial, all else zeroed)

In [ ]:
import time
WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)

ORIG_X = [d.x.clone() for d in dataset]
def apply_mask(mask):
    for d, x0 in zip(dataset, ORIG_X):
        t = x0[:, N_PRE:].clone()
        keep = torch.from_numpy(mask)
        t[:, ~keep] = 0.0
        d.x = torch.cat([x0[:, :N_PRE], t], dim=1)
def restore():
    for d, x0 in zip(dataset, ORIG_X):
        d.x = x0.clone()

BEST_LAMBDA = 0.1
arm_models = {name: {} for name in MASKS}
t0 = time.time()
for arm_name, mask in MASKS.items():
    apply_mask(mask)
    for seed in range(8):
        m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=seed)
        v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
        arm_models[arm_name][seed] = m
        torch.save(m.state_dict(), WORK/f"cache/{arm_name}_seed{seed}.pt")
        print(f"  [{arm_name}] seed {seed}: val PR {val_pr:.4f}  "
              f"val DCC {v.get('dcc',float('nan')):.4f}  [{(time.time()-t0)/60:.1f}m]")
    restore()

print(f"\ntrained {sum(len(v) for v in arm_models.values())} models across {len(MASKS)} arms")

## Cell 6 — Also load Phase 15/18's full and no_topo checkpoints (no retraining)

In [ ]:
def load_existing(prefix, seeds):
    out = {}
    for seed in seeds:
        cands = []
        for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
            if r.exists(): cands.extend(r.rglob(f"{prefix}{seed}.pt"))
        if not cands: continue
        state = torch.load(max(cands, key=lambda p: p.stat().st_size), map_location=DEVICE)
        m = LocalizingEGNN(N_PRE, TOPO_DIM).to(DEVICE); m.load_state_dict(state); m.eval()
        out[seed] = m
    return out

arm_models["full"] = load_existing("loc_deploy_seed", range(8))
arm_models["no_topo"] = load_existing("notopo_deploy_seed", range(8))
print(f"loaded {len(arm_models['full'])} 'full' + {len(arm_models['no_topo'])} 'no_topo' "
      f"reference checkpoints (up to 8 seeds each, from Phase 21/22's already-trained pool)")

## Cell 7 — Load Phase 15's cached benchmark features (no re-fetch)

In [ ]:
with open(locate("phase15_raw_features.pkl"), "rb") as fh:
    raw_cache = pickle.load(fh)
print(f"loaded cached features for {len(raw_cache):,} chains (COACH420 + HOLO4K)")
mu_b, sd_b = blob6b["topo_mean"], blob6b["topo_std"]

## Cell 8 — Score every arm on the cached benchmark

In [ ]:
from collections import defaultdict

@torch.no_grad()
def predict(entry, model_dict, mask=None):
    x = torch.from_numpy(entry["x"]).clone()
    if mask is None:
        x[:, N_PRE:] = 0.0                              # no_topo
    else:
        t = x[:, N_PRE:].clone()
        keep = torch.from_numpy(mask) if mask is not True else torch.ones(t.size(1), dtype=torch.bool)
        t = (t - torch.from_numpy(mu_b)) / torch.from_numpy(sd_b)   # standardize first
        t[:, ~keep] = 0.0    # THEN zero masked columns -- matches training-time apply_mask,
                              # which zeroed an ALREADY-standardized d.x (see Cell 5)
        x[:, N_PRE:] = t
    d = type("D", (), {})()
    d.x=x.to(DEVICE); d.pos=torch.from_numpy(entry["pos"]).to(DEVICE)
    d.edge_index=torch.from_numpy(entry["edge_index"]).to(DEVICE)
    d.edge_attr=torch.from_numpy(entry["edge_attr"]).to(DEVICE)
    d.batch = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
    offset = entry["ca"].mean(axis=0)
    seed_logits, seed_centers = [], []
    for m in model_dict.values():
        lg, pc = m(d)
        seed_logits.append(torch.sigmoid(lg).cpu().numpy())
        seed_centers.append(pc.cpu().numpy()[0])
    return float(np.mean(seed_logits, axis=0).sum()), np.mean(seed_centers, axis=0) + offset


def dcc(pred_center, ligs):
    if pred_center is None or not ligs: return None
    best = np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        best = min(best, float(np.linalg.norm(pred_center - lig_atoms.mean(axis=0))))
    return best if np.isfinite(best) else None


by_structure = defaultdict(list)
for key, entry in raw_cache.items():
    by_structure[key[:2]].append((key[2], entry))

ARM_MASK_FOR_SCORING = {**{k: v for k, v in MASKS.items()}, "full": True, "no_topo": None}

t0 = time.time()
print(f"{'arm':<14}{'DCC':>10}{'n':>6}")
print("-"*30)
summary = {}
for arm_name, mask in ARM_MASK_FOR_SCORING.items():
    models = arm_models[arm_name]
    hits, n = 0, 0
    for key, chain_list in by_structure.items():
        ligs = raw_cache[(key[0], key[1], chain_list[0][0])]["ligs"]
        best_conf, best_center = -np.inf, None
        for cid, entry in chain_list:
            conf, center = predict(entry, models, mask)
            if conf > best_conf: best_conf, best_center = conf, center
        d_ = dcc(best_center, ligs)
        if d_ is None: continue
        n += 1; hits += int(d_ < 4.0)
    rate = hits / max(n, 1)
    summary[arm_name] = {"dcc": rate, "n": n}
    print(f"{arm_name:<14}{rate:>10.4f}{n:>6}")
print(f"\nscored all {len(ARM_MASK_FOR_SCORING)} arms in {(time.time()-t0)/60:.1f} min")

## Cell 9 — Which piece recovers the benefit?

In [ ]:
base = summary["no_topo"]["dcc"]
full = summary["full"]["dcc"]
total_gain = full - base

print(f"no_topo baseline: {base:.4f}")
print(f"full (all topo):  {full:.4f}  (total gain: {total_gain:+.4f})\n")

print(f"{'arm':<14}{'DCC':>10}{'gain over no_topo':>20}{'% of full gain':>18}")
print("-"*62)
for name in ("H0_only", "H1_only", "H2_only", "burial_only"):
    g = summary[name]["dcc"] - base
    pct = 100 * g / total_gain if total_gain > 1e-9 else float("nan")
    print(f"{name:<14}{summary[name]['dcc']:>10.4f}{g:>+20.4f}{pct:>17.1f}%")

winner = max(("H0_only","H1_only","H2_only","burial_only"), key=lambda n: summary[n]["dcc"])
print(f"\nlargest single-piece recovery (8-seed ensemble): {winner} ({summary[winner]['dcc']:.4f})")

# --- per-seed breakdown for the two closest arms: is the gap real, or noise? ---
print(f"\n{'='*70}\nPER-SEED CHECK: is the top-vs-runner-up gap real, or seed noise?\n{'='*70}")
ranked = sorted(("H0_only","H1_only","H2_only","burial_only"), key=lambda n: -summary[n]["dcc"])
top_name, second_name = ranked[0], ranked[1]

def per_seed_dcc(arm_name):
    models_all = arm_models[arm_name]
    out = []
    for seed in sorted(models_all):
        single = {seed: models_all[seed]}
        hits, n = 0, 0
        for key, chain_list in by_structure.items():
            ligs = raw_cache[(key[0], key[1], chain_list[0][0])]["ligs"]
            best_conf, best_center = -np.inf, None
            for cid, entry in chain_list:
                conf, center = predict(entry, single, ARM_MASK_FOR_SCORING[arm_name])
                if conf > best_conf: best_conf, best_center = conf, center
            d_ = dcc(best_center, ligs)
            if d_ is None: continue
            n += 1; hits += int(d_ < 4.0)
        out.append(hits / max(n, 1))
    return np.array(out)

top_seeds = per_seed_dcc(top_name)
second_seeds = per_seed_dcc(second_name)
print(f"{top_name}: per-seed {np.round(top_seeds,4).tolist()}")
print(f"  mean {top_seeds.mean():.4f}  std {top_seeds.std():.4f}")
print(f"{second_name}: per-seed {np.round(second_seeds,4).tolist()}")
print(f"  mean {second_seeds.mean():.4f}  std {second_seeds.std():.4f}")
diff = top_seeds.mean() - second_seeds.mean()
noise = max(top_seeds.std(), second_seeds.std())
print(f"\ndiff: {diff:+.4f}  vs noise scale {noise:.4f}  ({abs(diff)/max(noise,1e-9):.2f}x)")
verdict = (f"{top_name} CONFIRMED as the leading contributor (beyond noise)" if abs(diff) > 1.5*noise
          else f"{top_name} vs {second_name} gap is WITHIN plausible seed noise -- not confidently resolved")
print(f"VERDICT: {verdict}")

json.dump({"summary": summary, "base": base, "full": full, "total_gain": total_gain,
           "winner": winner, "top_name": top_name, "second_name": second_name,
           "top_seeds": top_seeds.tolist(), "second_seeds": second_seeds.tolist(),
           "verdict": verdict},
          open(WORK/"phase23_summary.json", "w"), indent=2)
print("\nsaved -> phase23_summary.json")

---

## Reading this

This is a **3-seed first pass**, not yet validated at the rigor Phase 22 gave
the headline finding — treat the winner as a hypothesis to confirm with more
seeds, not a final answer, exactly as Phase 18's single-pass result was
treated before Phase 22 checked it properly.

- **If H2 wins clearly**: the cavity-shape intuition is right, and "the model
  benefits from features that directly describe enclosed voids, consistent
  with what a binding pocket physically is" is a mechanistic, defensible claim
  — not just a correlation.
- **If H1 or burial wins instead**: equally interesting, and worth saying
  plainly rather than forcing the tidier H2 story — it would mean the benefit
  comes from access-channel topology or simple depth, not cavity enclosure.
- **If no single piece recovers much of the gain**: the benefit may be
  distributed/synergistic across all three homology dimensions together,
  which is itself a finding (topology's value here isn't reducible to one
  "most important" feature).